### Libraries 

In [3]:
!pip install openai

In [4]:
import pandas as pd
import numpy as np
# from tqdm import tqdm
import random
import time
import csv
import json
import openai
import os
from tqdm import tqdm
import time

### Dataset loading and analysis

#### Arguments

In [5]:
arguments = pd.read_csv("arguments.csv")
arguments.head()

,Argument ID,Part,Usage,Conclusion,Stance,Premise
0,A01001,usa,train,Entrapment should be legalized,in favor of,if entrapment can serve to more easily capture...
1,A01002,usa,train,We should ban human cloning,in favor of,we should ban human cloning as it will only ca...
2,A01003,usa,train,We should abandon marriage,against,marriage is the ultimate commitment to someone...
3,A01004,usa,train,We should ban naturopathy,against,it provides a useful income for some people
4,A01005,usa,train,We should ban fast food,in favor of,fast food should be banned because it is reall...


In [6]:
count = 0
for conclusion, stance, premise in arguments[["Conclusion", "Stance", "Premise"]].tail(10).itertuples(index=False, name=None):
    if count == 3:
        break
    print("C: {}, S: {}, P: {}".format(conclusion, stance, premise))
    count+=1
    

C: Reservations in India should be based on economic status, S: against, P: In India, it's easy to hide some income resources and to get a false income certificate. Though the situations are changing for the better, it is not that difficult.
C: There should be a ban on Bandhs, S: in favor of, P: Industries are also one of the worst hits. Investors may lose interest to invest in manufacturing sector as bandh by anyone results in the loss to them.
C: There should be a ban on Bandhs, S: in favor of, P: Though right to protest is a fundamental right, people do not have right to take away the fundamental rights of others such as going to their work, doing their personal works etc. Not everyone is against to the changes by government. Some people may support it and may not support the ban. Forcing them to stay at home is undemocratic.


#### Subsetting arguments from USA - case of study

In [7]:
arguments_usa = arguments[arguments["Part"] == "usa"]
print("Dataframe with USA instances has {} arguments".format(arguments_usa.shape[0]))

Dataframe with USA instances has 5020 arguments


#### Values

In [8]:
import json

with open("values.json", "r", encoding="utf-8") as f:
    values_dict = json.load(f)
values_l1_in_json = {item["name"] for item in values_dict["values"]} # value names in json file - level 1

ground_truth = pd.read_csv("labels-level1.csv")
df_value_columns = set(ground_truth.columns[1:])  # skip first column (Argument ID)

# all values in json correspond to the 54 value categories from the ground truth


In [9]:
def build_values_prompt_text(values_dict):
    lines = []

    for v in values_dict["values"]:
        name = v["name"]
        vid = v["id"]
        desc = "; ".join(v["descriptions"]) 

        line = f'{name} - {desc}'
        lines.append(line)

    # return '\n'.join(lines)
    return lines

VALUES_LIST = build_values_prompt_text(values_dict)

# len(VALUES_LIST)

## Baseline Predictor of Values (1000 arguments)

In [10]:
arguments_503 = pd.read_csv("arguments_503.csv")
# we want to predict the same 503 arguments as BERT and SVM from the reference paper
l_503 = arguments_503['Argument ID'].to_list()
# some more arguments that are not part of the 503 arguments above, to analyse a bigger sample
l_497 = (
    arguments_usa.loc[~arguments_usa["Argument ID"].isin(l_503), "Argument ID"]
    .head(497)
    .tolist()
)
l_1000 = l_503 + l_497
print(f"Hay un total de {len(l_1000)} argumentos a analizar. Sin repeticiones, {len(set(l_1000))}.")
 # comprobar que no hay repeticiones en los argumentos

arguments_1000 = arguments_usa.loc[
    arguments_usa["Argument ID"].isin(l_503) |
    arguments_usa["Argument ID"].isin(l_497)
]

arguments_1000

Hay un total de 1000 argumentos a analizar. Sin repeticiones, 1000.


,Argument ID,Part,Usage,Conclusion,Stance,Premise
0,A01001,usa,train,Entrapment should be legalized,in favor of,if entrapment can serve to more easily capture...
1,A01002,usa,train,We should ban human cloning,in favor of,we should ban human cloning as it will only ca...
2,A01003,usa,train,We should abandon marriage,against,marriage is the ultimate commitment to someone...
3,A01004,usa,train,We should ban naturopathy,against,it provides a useful income for some people
4,A01005,usa,train,We should ban fast food,in favor of,fast food should be banned because it is reall...
...,...,...,...,...,...,...
5015,A25476,usa,test,We should oppose collectivism,in favor of,opposing collectivism is the right thing to do...
5016,A25480,usa,test,We should limit judicial activism,against,activist judges are an important part of bring...
5017,A25486,usa,test,We should subsidize student loans,in favor of,graduates should not have to start their caree...
5018,A25493,usa,test,We should oppose collectivism,against,collectivism is society working for the benefi...


##### Check if the LLM call works

In [11]:
# import base_url and api_key from private files 
from pathlib import Path

file_path = Path('Cognitive-Bias-and-Heuristics-in-Value-Identification\baseline_predictor.ipynb').resolve().parent.parent / "keys.txt"

with open(file_path, "r", encoding="utf-8") as f:
    first_two_lines = [next(f).rstrip("\n") for _ in range(2)]

base_url, api_key = first_two_lines[0].strip(), first_two_lines[1].strip()

# !pip install openai

from openai import OpenAI
def crear_cliente():
    client = OpenAI(
        base_url=base_url,
        api_key=api_key, 
    )
    return client
 
def call_llm(client, model, prompt):
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model=model
    )
    return response.choices[0].message.content


In [12]:
client = crear_cliente()
call_llm(client, 'gpt-oss-120b', 'qué es la IA generativa')

'**IA generativa** (Inteligencia Artificial Generativa) es una rama de la IA cuyo objetivo es crear **contenidos originales** a partir de datos de entrenamiento. En lugar de limitarse a clasificar, predecir o extraer información, estos sistemas **“generan”** texto, imágenes, audio, vídeo, código, estructuras químicas, etc., que nunca antes habían existido.\n\n---\n\n## 1. ¿Cómo funciona?\n\n| Paso | Descripción |\n|------|--------------|\n| **Recopilación de datos** | Se reúnen grandes volúmenes de ejemplos del tipo de contenido que se quiere generar (libros, fotos, piezas musicales, código). |\n| **Entrenamiento del modelo** | Un algoritmo (normalmente una red neuronal profunda) aprende a **modelar la distribución estadística** de esos datos. Aprende patrones, relaciones y estructuras internas. |\n| **Generación** | A partir de una “semilla” o **prompt** (texto, ruido, condición…) el modelo muestra una muestra de lo que ha aprendido, produciendo un nuevo artefacto que respeta la coher

#### Baseline Prediction - Values in Arguments

In [13]:
# len(['Be creative', 'Be curious', 'Have freedom of thought', 'Be choosing own goals', 'Be independent', 'Have freedom of action', 'Have privacy', 'Have an exciting life', 'Have a varied life', 'Be daring', 'Have pleasure', 'Be ambitious', 'Have success', 'Be capable', 'Be intellectual', 'Be courageous', 'Have influence', 'Have the right to command', 'Have wealth', 'Have social recognition', 'Have a good reputation', 'Have a sense of belonging', 'Have good health', 'Have no debts', 'Be neat and tidy', 'Have a comfortable life', 'Have a safe country', 'Have a stable society', 'Be respecting traditions', 'Be holding religious faith', 'Be compliant', 'Be self-disciplined', 'Be behaving properly', 'Be polite', 'Be honoring elders', 'Be humble', 'Have life accepted as is', 'Be helpful', 'Be honest', 'Be forgiving', 'Have the own family secured', 'Be loving', 'Be responsible', 'Have loyalty towards friends', 'Have equality', 'Be just', 'Have a world at peace', 'Be protecting the environment', 'Have harmony with nature', 'Have a world of beauty', 'Be broadminded', 'Have the wisdom to accept others', 'Be logical', 'Have an objective view'])

In [14]:
VALUE_PREDICTION_PROMPT = """
INSTRUCTIONS:
- Select for the given argument which of 54 justifications one could provide for it. 
- Typically, one could provide at least 1 and not more than 5 of these justifications for an argument. If you would select more than 10 justifications for an argument, reduce your selection to the most fitting ones.
- Make sure you understand the examples given.
- Read the argument and justification. Select 1 (someone could provide the justification for the argument, even if you may disagree) or 0 (the justification makes no sense for the argument).

EXAMPLES:
Example arguments against "Social media should be banned".

- Argument: We have to be honest. Social media does not make people polite. But it makes our lives easier and more interesting.	
- Select all justifications one could provide: (✔ have a comfortable life) (from "easier lives"), (✔ have pleasure) (also from "easier lives"), (✔  ahaven exciting life) (from "more interesting"), (✔ have a varied life) (also from "more interesting"). But do not select justifications for concessions ((✖ be polite)) or empty phrases ((✖ be honest), (✖ be logical), (✖ have an objective view) for "We have to be honest").
Example arguments in favor of "Social media should be banned".
- Argument: Through social media people can spread biased opinions on topics or misinform the general public.	
- Use the examples for each justification to get a better understanding of the justifications ((✔ have freedom of thought) from reduced misleading influence on people's thoughts). But do not select justifications only because they are connected to the topic in general ((✖ have privacy) for the general threat of social media to privacy: it is not mentioned here).

TASK:
Select for the given argument which of 54 justifications one could provide for it. 

Imagine someone is arguing "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
If asked 'Why is this good?', might this be their justification? "Because it is good to "VALUE"".

Complete this task for each of the justifications, that is, each of the 54 human values with their descriptions in the following list.

Justifications (in fixed order): "{VALUES_LIST}"


Return format:
- Return a dictionary of EXACTLY 54 keys and 54 values. Keys correspond to each of the justifications, following the fixed order. The dictionary values must be an integer: 0 or 1.
    - 1 = YES (someone could provide the justification for the argument)
    - 0 = NO (the justification makes no sense for the argument)
- The order MUST match exactly the order of the VALUES list: ['Be creative', 'Be curious', 'Have freedom of thought', 'Be choosing own goals', 'Be independent', 'Have freedom of action', 'Have privacy', 'Have an exciting life', 'Have a varied life', 'Be daring', 'Have pleasure', 'Be ambitious', 'Have success', 'Be capable', 'Be intellectual', 'Be courageous', 'Have influence', 'Have the right to command', 'Have wealth', 'Have social recognition', 'Have a good reputation', 'Have a sense of belonging', 'Have good health', 'Have no debts', 'Be neat and tidy', 'Have a comfortable life', 'Have a safe country', 'Have a stable society', 'Be respecting traditions', 'Be holding religious faith', 'Be compliant', 'Be self-disciplined', 'Be behaving properly', 'Be polite', 'Be honoring elders', 'Be humble', 'Have life accepted as is', 'Be helpful', 'Be honest', 'Be forgiving', 'Have the own family secured', 'Be loving', 'Be responsible', 'Have loyalty towards friends', 'Have equality', 'Be just', 'Have a world at peace', 'Be protecting the environment', 'Have harmony with nature', 'Have a world of beauty', 'Be broadminded', 'Have the wisdom to accept others', 'Be logical', 'Have an objective view']
- Do NOT skip any value and do NOT add text, explanation, or formatting

Example output as dictionary:
'Be creative':1, 'Be curious':0, 'Have freedom of thought':0, 'Be choosing own goals':0,..., 'Have the wisdom to accept others':1, 'Be logical':0, 'Have an objective view':0
(with the length of the list being EXACTLY 54 and the order matching the VALUES list)
"""


In [15]:
def run_experiment(client, model, arguments, output_path):
    if os.path.exists(output_path):
        existing_df = pd.read_csv(output_path)
        done_ids = set(existing_df["Argument ID"].tolist())
        results = existing_df.values.tolist()
        print(f"Resuming... {len(done_ids)} arguments already processed.")
    else:
        done_ids = set()
        results = []

    segons_llista = []

    for _, row in tqdm(arguments.iterrows(), total=len(arguments)):
        arg_id = row["Argument ID"]

        if arg_id in done_ids:
            continue

        start = time.time()
        
        prompt = VALUE_PREDICTION_PROMPT.format(
            CONCLUSION=row["Conclusion"],
            STANCE=row["Stance"],
            PREMISE=row["Premise"],
            VALUES_LIST=VALUES_LIST
        )

        vector = call_llm(client, model, prompt)

        end = time.time()
        segons = end - start
        segons_llista.append(segons)

        results.append([arg_id] + [str(vector)] + [segons])

        df = pd.DataFrame(
            results,
            columns=["Argument ID"] + ["values"] + ["seconds"]
        )

        df.to_csv(output_path, index=False)

    return pd.DataFrame(results, columns=["Argument ID"] + ["seconds"])

In [16]:
VALUES_LIST

['Be creative - allowing for more creativity or imagination; being more creative; fostering creativity; promoting imagination',
 'Be curious - being the more interesting option; fostering curiosity; making people more keen to learn; promoting discoveries; sparking interest',
 "Have freedom of thought - allowing people to figure things out on their own; allowing people to make up their mind; resulting in less censorship; resulting in less influence on people's thoughts",
 'Be choosing own goals - allowing people to choose what is best for them; allowing people to decide on their life; allowing people to follow their dreams',
 'Be independent - allowing people to plan on their own; resulting in fewer times people have to ask for consent',
 'Have freedom of action - allowing people to be self-determined; allowing people to do things even though this may hurt them in the long run; resulting in more times people can do what they want',
 'Have privacy - allowing for private spaces; allowing 

### LLM selection

Si el objetivo es seguir instrucciones de texto (instruction following), sin necesidad de programar o analizar imágenes, el orden aproximado sería:
* **Qwen3.6-35B-A3B**	⭐⭐⭐⭐⭐	Uno de los mejores modelos abiertos actuales para seguir instrucciones, razonamiento y conversación. https://huggingface.co/Qwen/Qwen3.6-35B-A3B // https://qwen.ai/blog?id=qwen3.6-35b-a3b

* **Llama 3.3 70B**	⭐⭐⭐⭐⭐	Excelente capacidad conversacional e instruction following. Muy cercano a Qwen, aunque suele ser algo menos preciso en tareas complejas. https://ollama.com/library/llama3.3:70b

* **GPT-OSS-120B**	⭐⭐⭐⭐☆	Muy potente por tamaño (120B), pero en la práctica no siempre supera a Qwen 3.6 en adherencia estricta a instrucciones. https://openai.com/es-ES/index/introducing-gpt-oss/

In [ ]:
llms = ['gpt-oss-120b', 'llama3.3:70b', 'Qwen3.6-35B-A3B']
output_csv = ['results/baseline_gpt_oss_1000.csv','results/baseline_llama_1000.csv','results/baseline_qwen_1000.csv']

#### Run Experiments

In [22]:
client = crear_cliente()
for model, output in zip(llms, output_csv):
    results = run_experiment(client, model, arguments_1000, output)

  0%|          | 0/1000 [00:00<?, ?it/s]


PermissionDeniedError: Error code: 403 - {'error': {'message': "team not allowed to access model. This team can only access models=['poligpt', 'nomic-embed-text', 'text-embedding-3-large', 'text-embedding-ada-002', 'llama', 'llama-mini', 'qwen', 'qwen-mini', 'gpt-oss-120b', 'gpt-4.1', 'gpt-4.1-mini', 'gpt-4.1-nano', 'gpt-5', 'gpt-5-mini', 'gpt-5-nano', 'gpt-audio', 'gpt-audio-mini', 'gpt-4o']. Tried to access Qwen3.6-35B-A3B", 'type': 'team_model_access_denied', 'param': 'model', 'code': '403'}}